In [9]:
import time
import torch
import torch.nn as nn

print("=" * 60)
print("1. VS Code -> Colab 云端连接与硬件自检")
print("=" * 60)
assert torch.cuda.is_available(), "CUDA 不可用！请检查右上角内核是否已选择 Colab (L4)"

device = torch.device("cuda:0")
props = torch.cuda.get_device_properties(device)
print(f"连接成功！分配显卡 : {props.name}")
print(f"显存容量           : {props.total_memory / (1024**3):.2f} GB")
print(f"CUDA / PyTorch     : {torch.version.cuda} / {torch.__version__}")

# 开启 Ada Lovelace (CC 8.9) 架构的 TF32 硬件加速
torch.set_float32_matmul_precision("high")

# 2. BF16 混合精度矩阵算力基准测试
n = 8192
a = torch.randn(n, n, device=device, dtype=torch.bfloat16)
b = torch.randn(n, n, device=device, dtype=torch.bfloat16)
for _ in range(10):
    _ = torch.matmul(a, b)
torch.cuda.synchronize()

t0 = time.perf_counter()
for _ in range(50):
    _ = torch.matmul(a, b)
torch.cuda.synchronize()
dt = (time.perf_counter() - t0) / 50
bf16_tflops = (2 * (n ** 3)) / (dt * 1e12)
print(f"BF16 混合精度算力  : {bf16_tflops:.2f} TFLOPS (单次 {dt*1000:.2f} ms)")

# 3. 强化学习策略网络训练闭环实测 (Batch=4096)
model = nn.Sequential(
    nn.Linear(64, 512), nn.ELU(),
    nn.Linear(512, 512), nn.ELU(),
    nn.Linear(512, 512), nn.ELU(),
    nn.Linear(512, 12)
).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
obs = torch.randn(4096, 64, device=device)
act = torch.randn(4096, 12, device=device)

for _ in range(20):
    optimizer.zero_grad()
    loss = ((model(obs) - act) ** 2).mean()
    loss.backward()
    optimizer.step()
torch.cuda.synchronize()

steps = 300
t0 = time.perf_counter()
for _ in range(steps):
    optimizer.zero_grad()
    loss = ((model(obs) - act) ** 2).mean()
    loss.backward()
    optimizer.step()
torch.cuda.synchronize()
elapsed = time.perf_counter() - t0

print("=" * 60)
print(f"RL 策略训练更新频率: {steps / elapsed:,.1f} 步/秒")
print(f"并行样本吞吐量     : {(steps * 4096) / elapsed:,.0f} 样本/秒")
print("=" * 60)

1. VS Code -> Colab 云端连接与硬件自检
连接成功！分配显卡 : NVIDIA L4
显存容量           : 22.03 GB
CUDA / PyTorch     : 13.0 / 2.14.1+cu130
BF16 混合精度算力  : 51.34 TFLOPS (单次 21.42 ms)
RL 策略训练更新频率: 574.5 步/秒
并行样本吞吐量     : 2,353,066 样本/秒


In [8]:
print(1 + 1)

2


In [10]:
from google.colab import drive
from pathlib import Path
import os

# 挂载 Google Drive，已经挂载时会复用
drive.mount("/content/drive")

# 在 /content 下建立 MyDrive 快捷入口
shortcut = Path("/content/MyDrive")
if not os.path.lexists(shortcut):
    shortcut.symlink_to(
        "/content/drive/MyDrive",
        target_is_directory=True,
    )

# 后续代码默认在这个项目目录中读写文件
project = shortcut / "云端算力测试"
os.chdir(project)

print("当前工作目录", Path.cwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
当前工作目录 /content/drive/MyDrive/云端算力测试


In [11]:
from pathlib import Path

print("当前目录", Path.cwd())
print("目录内容", [p.name for p in Path.cwd().iterdir()][:10])

当前目录 /content/drive/MyDrive/云端算力测试
目录内容 ['测试算力1.ipynb']
